In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [3]:
# 加载模型与TOkenizer
from transformers import AutoModelForCausalLM,AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:00<00:00, 11454.31it/s]


In [4]:
# 数据集 处理数据集至openai格式
from datasets import load_dataset
dataset_dict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl",
                                              "test":"data/keywords_data_test.jsonl"})
# 转成openai格式
def map_func(exapmle):
    conversation = exapmle["conversation"]
    messages=[]
    for item in conversation:
        messages.append({"role":"user","content":item["human"]})
        messages.append({"role":"assistant","content":item["assistant"]})
    return {"messages":messages}

dataset_dict=dataset_dict.map(map_func,batched=False,remove_columns=["conversation_id","category","conversation","dataset"])

In [5]:
dataset_dict["train"][0]

{'messages': [{'role': 'user',
   'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词'},
  {'role': 'assistant', 'content': '高氟铍矿;配料;熔炼;回收率;脱氟率'}]}

In [6]:
from trl import SFTConfig,SFTTrainer
training_args = SFTConfig(
    output_dir="/home/tianjp/llmLearn/stf/Qwen3-0.6B/sft-full",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_total_limit=2,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    assistant_only_loss=True,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
)


In [7]:
dataloader = trainer.get_train_dataloader()
batch=next(iter(dataloader))

In [8]:
batch["input_ids"].shape

torch.Size([4, 205])

In [11]:
print(tokenizer.decode(batch["input_ids"][0]))

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant
<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,1.192725,1.433586
200,1.328960,1.376793
300,1.303247,1.339827
400,1.232789,1.295498
500,1.340603,1.266861
600,1.188424,1.259229
700,1.117151,1.231079
800,1.143010,1.221223
900,1.196132,1.211916
1000,1.067395,1.210619


Writing model shards: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.27s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1000, training_loss=1.2650289964675903, metrics={'train_runtime': 489.5829, 'train_samples_per_second': 8.17, 'train_steps_per_second': 2.043, 'total_flos': 2798465934950400.0, 'train_loss': 1.2650289964675903})

In [13]:
trainer.save_model("/home/tianjp/llmLearn/stf/Qwen3-0.6B/sft-full/best")

Writing model shards: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.31s/it]


In [14]:
next(model.parameters()).dtype

torch.bfloat16

In [15]:
next(trainer.parameters()).dtype

AttributeError: 'SFTTrainer' object has no attribute 'parameters'